In [2]:
import pandas as pd
import re
import os
import sys
from datetime import datetime, timedelta
from plotnine import *

In [3]:
# This script is intended to add new samples
# Either from public databases (NCBI and GISAID)
# Or from Nicole K. 

In [4]:
# Import various metadatas

# metadata = pd.read_csv("./metadata/updated_metadata_cleaned.csv")
# gisaid = pd.read_excel("./database_pulls/2026-03-11/gisaid_epiflu_isolates.xls")
# ncbi = pd.read_csv("./database_pulls/2026-03-11/sequences.csv", sep = ",")
metadata = pd.read_csv("./metadata/updated_metadata_cleaned_may2026.tsv", sep = "\t")

gisaid = pd.read_excel("./database_pulls/gisaid_ncbi_2026-05-27/gisaid_epiflu_isolates.xls")
ncbi = pd.read_csv("./database_pulls/gisaid_ncbi_2026-05-27/sequences.csv", sep = ",")

In [13]:
print(f"Current Dataset: {len(metadata)}")
print(metadata["newid"].is_unique)
print(f"GISAID viruses: {len(gisaid)}")
print(f"NCBI sequences (all segments): {len(ncbi)}")

Current Dataset: 365
True
GISAID viruses: 532
NCBI sequences (all segments): 853


In [166]:
#Function: Read in the FASTA as a dataframe

def fasta_reader(path_to_fasta, output_name):
    fasta_data = []
    FASTANAME = path_to_fasta
        
    with open(FASTANAME) as f:
        header = ""
        sequence = ""
        for line in f:
            if line.startswith(">"):
                if header != "":
                    fasta_data.append({"header": header, "sequence": sequence})
                header = line.strip() 
                sequence = ""
            else:
                sequence += line.strip()
        fasta_data.append({"header": header, "sequence": sequence}) #last line 
        
    globals()[output_name] = pd.DataFrame(fasta_data)

    return

In [167]:
#Function: write a new FASTA with updated header (function written by Maria)  
def fasta_writer(path, filename, df, header):
            
    try:  
        os.mkdir(path)

    except OSError as error:
        pass

    with open(f"{path}{filename}", "w") as f:
        for index, row in df.iterrows():
            f.write(f"{row[header]}\n")
            f.write(f"{row['sequence']}\n")

In [168]:
def gisaid_identifiers(df):

    df_new = df.copy()
    
    if len(df_new.columns) == 2: # This is a FASTA file
        df_new["strainname"] = df_new["header"].str.split("|", expand = True)[2] # Get just the strain name
        if df_new["strainname"].str.contains(r"_[A-Z]{2}$").any(): # If it ends in the segment...
            df_new["strainname"] = [re.sub(r"_[A-Z]{2}$", "", i) for i in df_new["strainname"]] # ...remove (you'll get it later)
        df_new["strainname"] = df_new["strainname"].str.replace(" ", "_") # Remove any spaces in the strain name
        df_new["id_code"] = df_new["header"].str.split("|", expand = True)[0]
        # df_new["strainname"].to_csv("./database_pulls/gisaid_ncbi_2026-05-26/gisaid_identifiers.tsv", sep = "\t")
    
    elif len(df_new.columns) > 2: # This is (likely) a metadata file
        df_new["id_code"] = df_new["Isolate_Id"]
        df_new["strainname"] = df_new["Isolate_Name"]
    return(df_new)

In [169]:
def ncbi_identifiers(df):

    df_new = df.copy()

    if len(df_new.columns) == 2: # This is a FASTA file
        df_new["strainname"] = df_new["header"].str.split("(", expand = True)[1] # The strain name
        df_new["strainname"] = df_new["strainname"].str.replace(" ", "_") # Remove any spaces in the strain name
        # df_new["id_code"] = df_new["header"].str.split("|", expand = True)[0].str.replace(".1 ", "")
        df_new["id_code"] = ">" + df_new["strainname"] # I wish the NCBI ID could be the accession, but those aren't shared across segments
    
    elif len(df_new.columns) > 2: # This is (likely) a metadata file
        df_new["strainname"] = df_new["GenBank_Title"].str.split("(", expand = True)[1] # The strain name
        # df_new["id_code"] = df_new["Accession"]
        df_new["id_code"] = df_new["strainname"]
        
    return(df_new)

In [170]:
# Step One: Determine which of these H5N5 sequences are A6

def filter_genotype(FASTA1, FASTA2): #, gisaid_semgents, ncbi_segments):

    # Read in the FASTAs
    fasta_reader(FASTA1, "gisaid_fasta") 
    fasta_reader(FASTA2, "ncbi_fasta")

    # Make an identifier and segment column
    gisaid_fasta2 = gisaid_identifiers(gisaid_fasta)
    gisaid_fasta2["segment"] = gisaid_fasta2["header"].str.rsplit("|", n=1).str[-1]

    ncbi_fasta2 = ncbi_identifiers(ncbi_fasta)
    ncbi_fasta2["segment"] = ncbi_fasta2["header"].str.extract(r"segment\s(\d{1})")

    # Divide FASTAs by segment
    pb2_gisaid = gisaid_fasta2[gisaid_fasta2["segment"] == "PB2"]
    pb1_gisaid = gisaid_fasta2[gisaid_fasta2["segment"] == "PB1"]
    pa_gisaid = gisaid_fasta2[gisaid_fasta2["segment"] == "PA"]
    ha_gisaid = gisaid_fasta2[gisaid_fasta2["segment"] == "HA"]
    np_gisaid = gisaid_fasta2[gisaid_fasta2["segment"] == "NP"]
    na_gisaid = gisaid_fasta2[gisaid_fasta2["segment"] == "NA"]
    mp_gisaid = gisaid_fasta2[gisaid_fasta2["segment"] == "MP"]
    ns_gisaid = gisaid_fasta2[gisaid_fasta2["segment"] == "NS"]

    pb2_ncbi = ncbi_fasta2.loc[ncbi_fasta2["segment"] == "1"].drop_duplicates(subset="id_code", keep="first")
    pb1_ncbi = ncbi_fasta2.loc[ncbi_fasta2["segment"] == "2"].drop_duplicates(subset="id_code", keep="first")
    pa_ncbi = ncbi_fasta2.loc[ncbi_fasta2["segment"] == "3"].drop_duplicates(subset="id_code", keep="first")
    ha_ncbi = ncbi_fasta2.loc[ncbi_fasta2["segment"] == "4"].drop_duplicates(subset="id_code", keep="first") 
    np_ncbi = ncbi_fasta2.loc[ncbi_fasta2["segment"] == "5"].drop_duplicates(subset="id_code", keep="first")
    na_ncbi = ncbi_fasta2.loc[ncbi_fasta2["segment"] == "6"].drop_duplicates(subset="id_code", keep="first")
    mp_ncbi = ncbi_fasta2.loc[ncbi_fasta2["segment"] == "7"].drop_duplicates(subset="id_code", keep="first")
    ns_ncbi = ncbi_fasta2.loc[ncbi_fasta2["segment"] == "8"].drop_duplicates(subset="id_code", keep="first")

    # Merge each segment
    pb2_concat_h5n5 = pd.concat([pb2_gisaid, pb2_ncbi], join = "outer")
    pb1_concat_h5n5 = pd.concat([pb1_gisaid, pb1_ncbi], join = "outer")
    pa_concat_h5n5 = pd.concat([pa_gisaid, pa_ncbi], join = "outer")
    ha_concat_h5n5 = pd.concat([ha_gisaid, ha_ncbi], join = "outer")
    np_concat_h5n5 = pd.concat([np_gisaid, np_ncbi], join = "outer")
    na_concat_h5n5 = pd.concat([na_gisaid, na_ncbi], join = "outer")
    mp_concat_h5n5 = pd.concat([mp_gisaid, mp_ncbi], join = "outer")
    ns_concat_h5n5 = pd.concat([ns_gisaid, ns_ncbi], join = "outer")
    # print(f"HA length before: {len(ha_concat_h5n5)}")

    fasta_dict = {"pb2": pb2_concat_h5n5, "pb1": pb1_concat_h5n5, "pa": pa_concat_h5n5, "ha": ha_concat_h5n5, "np": np_concat_h5n5, "na": na_concat_h5n5, "mp": mp_concat_h5n5, "ns": ns_concat_h5n5}

    # If there are duplicates, drop 'em
    # if ha_concat_h5n5["identifier"].is_unique == True:
    #     print("All unique HA identifiers")
    # else:
    #     print("Duplicates!")
    #     for key, df in fasta_dict.items():
    #         new_df = df.drop_duplicates(subset="identifier", keep="first", inplace=True) 
    #         fasta_dict[key].update(new_df)

    for key, df in fasta_dict.items():
        df["sequence"] = df["sequence"].str.upper() # Makes all sequences uppercase
        fasta_writer("./database_pulls/gisaid_ncbi_2026-05-27/", f"{key}.fasta", df, "id_code") # Print new FASTAs

    for key, df in fasta_dict.items():
        print(f"{key} length: {len(df)}")

    return(fasta_dict)

In [171]:
# concatenated_fastas = filter_genotype("./database_pulls/2026-03-11/gisaid_epiflu_sequence.fasta", "./database_pulls/2026-03-11/sequences.fasta")
concatenated_fastas = filter_genotype("./database_pulls/gisaid_ncbi_2026-05-27/gisaid_epiflu_sequence.fasta", "./database_pulls/gisaid_ncbi_2026-05-27/sequences.fasta")

pb2 length: 584
pb1 length: 582
pa length: 584
ha length: 603
np length: 591
na length: 597
mp length: 592
ns length: 591


In [172]:
# Count how many identifiers are shared across all segments

all_identifiers = []

for key, df in concatenated_fastas.items():
    ids = set(df["id_code"])
    all_identifiers.append(ids)

shared_identifiers = set.intersection(*all_identifiers)

print(len(shared_identifiers))

576


In [153]:
# Here, run the per-segment FASTA files you just make through GenoFlu to generate the results

In [173]:
genoflu_results = pd.read_csv("../GenoFLU-multi/h5n5_rerun_may2026/results/results.tsv", sep = "\t")

In [188]:
# Step Two: Generate a list of strain names that are not in the current metadata

def identify_new_public_sequences(current, gisaid, ncbi, genoflu_results):

    new_from_databases = []

    # Subset down to A6 only
    a6 = genoflu_results["Strain"].loc[genoflu_results["Genotype"] == "A6"].to_list()
    a6_gisaid = gisaid.loc[gisaid["Isolate_Name"].str.replace(" ", "_").isin(a6)]
    a6_ncbi = ncbi.loc[ncbi["GenBank_Title"].str.replace(" ", "_").str.split("(", expand = True)[1].isin(a6)]
    
    gisaid_strains = a6_gisaid["Isolate_Name"].to_list()
    ncbi_strains = set(a6_ncbi["GenBank_Title"].str.split("(", expand = True)[1].to_list())

    for i in gisaid_strains:
        if i in set(current["Isolate_Name"]):
            pass
        else:
            new_from_databases.append(i)

    print(ncbi_strains) # NCBI is a little difficult because the strain names were changed, but there are relatively few of them so I will add by hand
    
    print(f"Number of new sequences from GISAID: {len(new_from_databases)}")

    return(new_from_databases)

In [189]:
new_sequences = identify_new_public_sequences(metadata, gisaid, ncbi, genoflu_results)

{'A/Black Vulture/MA/25-021407-019-original/2025', 'A/Common Eider/MA/25-021407-028-original/2025', 'A/Great black-backed gull/MA/25-021407-026-original/2025', 'A/Turkey Vulture/CT/25-006834-003-original/2025', 'A/Common Raven/MA/25-019390-017-original/2025', 'A/Bufflehead/MA/25-007118-044-original/2025', 'A/Great Black-Backed Gull/MA/25-019390-018-original/2025', 'A/waterfowl/Russia/1526-4/2021', 'A/Bald Eagle/OH/25-005188-001-original/2025', 'A/Northern Fulmar/Germany-NI/2024AI04273/2024', 'A/Common Raven/MA/25-019390-016-original/2025', 'A/Polar Bear/AK/26G03248/2025', 'A/Washington/2148/2025', 'A/bald_eagle/Ohio/OH25-5320/2025', 'A/Great black-backed gull/MA/25-021407-025-original/2025', 'A/shelduck/Kalmykia/1814-1/2021', 'A/Red-breasted Merganser/MA/25-021407-029-original/2025', 'A/Turkey/WA/25G04434-001-v/2025'}
Number of new sequences from GISAID: 0


In [190]:
new_sequences.extend([
                      "A/Polar Bear/AK/26G03248/2025"
                     ])

In [191]:
print(new_sequences)

['A/Polar Bear/AK/26G03248/2025']


In [192]:
# Some of these have slightly updated location information on GenBank that was not in GISAID 
# And I think Alvin ended up pulling them from GISAID

update_metadata_ncbi = [
                        "A/common eider/USA/021407-028/2025", 
                        "A/great black-backed gull/USA/019390-018/2025", 
                        "A/common raven/USA/019390-017/2025",
                        "A/Bufflehead/MA/25-007118-044-original/2025", 
                        "A/bald eagle/OH/25-005188-001-original/2025",
                        "A/Common Raven/MA/25HP00745/2025", # This is under a different code in NCBI but Nicole's spreadsheet links them
                        "A/Great black-backed gull/MA/25-021407-026-original/2025",
                        "A/Black Vulture/MA/25-021407-019-original/2025"
                       ]

In [193]:
# Alright
# Now I have a list of sequences that are not in the metadata at all and need to be added
# So I pull their FASTA sequences from the concatenated FASTA file 
# And I pull their metadata
# And I concatenate to current metadata

In [194]:
def add_new_public_sequences(new, current, gisaid, ncbi, concatenated_fastas):

    # First pull the metadata from gisaid and ncbi
    new_from_gisaid = gisaid.loc[gisaid["Isolate_Name"].str.replace(" ", "_").isin(new)] # Some have underscores originally
    new_from_gisaid = gisaid.loc[gisaid["Isolate_Name"].isin(new)] # some have spaces originally
    new_from_ncbi = ncbi.loc[ncbi["GenBank_Title"].str.split("(", expand = True)[1].isin(new)].drop_duplicates(subset = "SRA_Accession", keep = "first")

    new_from_all = pd.concat([new_from_gisaid, new_from_ncbi])

    # Then get the metadata you want from this
    columns = ["Isolate_Id", "Isolate_Name", "Collection_Date", "Subtype", "Genotype", "Location", "Host", 
              "GenBank_Title", "Accession", "Country", "Segment"]
    new_metadata = pd.concat([current, new_from_all[columns]])
    print(f"Metadata goes from {len(current)} to {len(new_metadata)} samples")
    
    # Pull sequences from FASTAs
    new_fastas = {}
    
    for f in concatenated_fastas.values():
        segment = f.loc[f["segment"].astype(str).str.contains(r"[A-Za-z]"), "segment"].iloc[0]
        new_fastas[segment] = {}
        seq_to_pull = f.loc[(f["strainname"].str.replace(">", "").isin(new)) | (f["strainname"].str.replace(">", "").str.replace("_", " ").isin(new))]
        new_fastas[segment] = seq_to_pull

    return(new_metadata, new_fastas)

In [195]:
new_metadata, new_fastas = add_new_public_sequences(new_sequences, metadata, gisaid, ncbi, concatenated_fastas)

Metadata goes from 364 to 365 samples


In [185]:
# new_metadata.to_csv("./metadata/updated_metadata_uncleaned.tsv", sep = "\t")
new_metadata.to_csv("./metadata/updated_metadata_uncleaned_may2026.tsv", sep = "\t")

In [120]:
# Alternately, I can just add gisaid and ncbi sequences together 
# I do this because I want to start from scratch (with TPE GISAID sequences excluded)
# This function adds GISAID and NCBI together, but doesn't factor in any of the original sequences

def merge_gisaid_ncbi_from_scratch(gisaid, ncbi, concatenated_fastas, genoflu_results):

    gisaid_upt = gisaid_identifiers(gisaid)
    ncbi_upt = ncbi_identifiers(ncbi)

    ncbi_upt_deduped = ncbi_upt.drop_duplicates(subset="strainname", keep="first")
    print(f"NCBI with all segments: {len(ncbi_upt)}, NCBI deduped: {len(ncbi_upt_deduped)}")
    
    # Concatenate the gisaid and ncbi metadata
    concatenated_meta = pd.concat([gisaid_upt, ncbi_upt_deduped], axis = 0, join = "outer")
    print(f"{len(gisaid_upt)} + {len(ncbi_upt_deduped)} = {len(concatenated_meta)} sequences in concatenated metadata")

    # Using the genflu results and the metadata id_codes, limit to just A6
    A6 = genoflu_results["Strain"].loc[genoflu_results["Genotype"] == "A6"].to_list()
    print(f"{len(A6)} A6 genomes")
    concatenated_meta_a6 = concatenated_meta[concatenated_meta["id_code"].isin(A6)]
    print(f"{len(concatenated_meta_a6)} A6 sameples from GISAID and NCBI (not yet deduplicated)")

    # NOW look at the strain names, and see if you can deduplicate
    strains = concatenated_meta_a6["strainname"]
    # strains.to_csv("./database_pulls/gisaid_ncbi_2026-05-26/allstrains.tsv", sep = "\t")
    
    concatenated_meta_a6["from_ncbi"] = (~concatenated_meta_a6["Accession"].isna())
    concat_meta_a6_deduped = concatenated_meta_a6.sort_values("from_ncbi", ascending=False).drop_duplicates(subset="strainname", keep="first").drop(columns="from_ncbi")
    print(f"{len(concat_meta_a6_deduped)} A6 samples from GISAID and NCBI (deduplicated)")
    
    # Remove any strain names that don't really exist
    weird_strainname = concat_meta_a6_deduped.loc[~concat_meta_a6_deduped["strainname"].str.contains(r"^A/", na = False)]
    final_meta = concat_meta_a6_deduped[~weird_strainname]
    print(f"Final A6 sequences from GISAID and NCBI Virus: {len(final_meta)}")

    # Pull sequences from FASTAs
    new_fastas = {}
    
    for f in concatenated_fastas.values():
        segment = f.loc[f["segment"].astype(str).str.contains(r"[A-Za-z]"), "segment"].iloc[0]
        new_fastas[segment] = {}
        seq_to_pull = f.loc[(f["identifier"].str.replace(">", "").isin(new)) | (f["identifier"].str.replace(">", "").str.replace("_", " ").isin(new))]
        new_fastas[segment] = seq_to_pull

    return(final_meta, new_fastas)

In [ ]:
merge_gisaid_ncbi_from_scratch(gisaid, ncbi, concatenated_fastas, genoflu_results)

In [315]:
for segment, df in new_fastas.items():
    label = str(segment)
    fasta_writer("./alignments/", f"{label}_fastas_to_add.fasta", df)

In [482]:
# Function made with help from Chat GPT

def add_new_tufts_sequences(strains, metadata):

    # Read in FASTAs and extract the Tufts code
    fasta_reader("./database_pulls/Nicole_2026-03-12/25ne00305COEIconsensus.fasta", "NE00305")
    fasta_reader("./database_pulls/Nicole_2026-03-12/25HP00818GBBGconsensus.fasta", "HP00818")
    fasta_reader("./database_pulls/Nicole_2026-03-12/24WP00294BLVUconsensus.fasta", "WP00294")
    fasta_reader("./database_pulls/Nicole_2026-03-12/24WI00326HERGconsensus.fasta", "WI00326")
    fasta_reader("./database_pulls/Nicole_2026-03-12/24WI00314Sanderlingconsensus.fasta", "WI00314")
    fasta_reader("./database_pulls/Nicole_2026-03-12/24WI00282GBBGconsensus.fasta", "WI00282")
    fasta_reader("./database_pulls/Nicole_2026-03-12/23NE01733GBBGconsensus.fasta", "NE01733")
    
    list_of_fastas = [NE00305, HP00818, WP00294, WI00326, WI00314, WI00282, NE01733]
    
    # Read in metadata and filter to just those in the list
    tufts_meta = pd.read_excel(metadata)
    tufts_meta_filtered = tufts_meta[tufts_meta["Tufts sample ID"].isin(strains)]
    
    # Merge on Tufts code
    merged_fastas = {}
    
    for fasta in list_of_fastas:
        # Extract a Tufts sample ID for the FASTA to match to Tufts metadata
        fasta["Tufts sample ID"] = fasta["header"].str.extract(r"(\d{2}\w{2}\d{5})")
        fasta["Tufts sample ID"] = fasta["Tufts sample ID"].str.upper()
        fasta["segment"] = fasta["header"].str.split("|", expand = True)[3]

        fasta_meta = fasta.merge(tufts_meta_filtered, on = "Tufts sample ID", how = "inner")

        # Generate an ID so you can match to your own metadata to add headers later on
        fasta_meta["id"] = ">" + fasta_meta["Tufts sample ID"].str.split("-", expand = True)[0]
        fasta_meta.loc[fasta_meta["NVSL sample ID"].str.contains(r"(\d{2}\-\d{6}\-\d{3})", regex = True), "id"] = ">" + fasta_meta["NVSL sample ID"]
        
        for code in fasta_meta["Tufts sample ID"].unique():

            sample_df = fasta_meta[fasta_meta["Tufts sample ID"] == code]

            if code not in merged_fastas:
                merged_fastas[code] = {}

            for segment in sample_df["segment"].unique():
                segment_fasta = sample_df[sample_df["segment"] == segment]

                merged_fastas[code][segment] = segment_fasta

    # Concatenate each segment across codes
    segments_combined = {}

    for code, seg_dict in merged_fastas.items():
        for segment, df in seg_dict.items():
            segments_combined.setdefault(segment, []).append(df)

    # concatenate
    segments_combined = {segment: pd.concat(dfs, ignore_index=True) for segment, dfs in segments_combined.items()}


    return(segments_combined)

In [483]:
fasta_list = []
strains = ["23NE01733","24WI00282","24WI00314","24WI00326","24WP00294","25HP00818","25NE00305"]
meta = "./database_pulls/Nicole_2026-03-12/h5n5_missingseqs-fromnicole_filledin.xlsx"

tufts_sequences_by_segment = add_new_tufts_sequences(strains, meta)

/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_3912/2015219901.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_3912/2015219901.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_3912/2015219901.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_3912/2015219901.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_3912/2015219901.py:33: UserWarning: This pattern is interpreted as a regular expr

In [484]:
for segment, seg_df in tufts_sequences_by_segment.items():
    try:  
        os.mkdir("./alignments/2026-03-16/")

    except OSError as error:
        pass

    with open(f"./alignments/2026-03-16/{segment}_from_tufts.fasta", "w") as f:
        for index, row in seg_df.iterrows():
            f.write(f"{row['id']}\n") # save with your own ID to match headers
            f.write(f"{row['sequence']}\n")

In [493]:
# Function: this code is for adding the correct headers

def correct_headers(METADATA):
    
    # Required: updated_metadata_uncleaned and fastas
    
    # fasta_reader("./alignments/2026-03-16/HA_fastas_to_add.fasta", "fastaHA")
    # fasta_reader("./alignments/2026-03-16/MP_fastas_to_add.fasta", "fastaMP")
    # fasta_reader("./alignments/2026-03-16/NA_fastas_to_add.fasta", "fastaNA")
    # fasta_reader("./alignments/2026-03-16/NP_fastas_to_add.fasta", "fastaNP")
    # fasta_reader("./alignments/2026-03-16/NS_fastas_to_add.fasta", "fastaNS")
    # fasta_reader("./alignments/2026-03-16/PA_fastas_to_add.fasta", "fastaPA")
    # fasta_reader("./alignments/2026-03-16/PB1_fastas_to_add.fasta", "fastaPB1")
    # fasta_reader("./alignments/2026-03-16/PB2_fastas_to_add.fasta",  "fastaPB2")

    fasta_reader("./alignments/2026-03-16/HA_from_tufts.fasta", "fastaHA")
    fasta_reader("./alignments/2026-03-16/MP_from_tufts.fasta", "fastaMP")
    fasta_reader("./alignments/2026-03-16/NA_from_tufts.fasta", "fastaNA")
    fasta_reader("./alignments/2026-03-16/NP_from_tufts.fasta", "fastaNP")
    fasta_reader("./alignments/2026-03-16/NS_from_tufts.fasta", "fastaNS")
    fasta_reader("./alignments/2026-03-16/PA_from_tufts.fasta", "fastaPA")
    fasta_reader("./alignments/2026-03-16/PB1_from_tufts.fasta", "fastaPB1")
    fasta_reader("./alignments/2026-03-16/PB2_from_tufts.fasta",  "fastaPB2")

    metadata = pd.read_csv(METADATA, sep = ",")

    dictionary = {"PB2": fastaPB2, "PB1": fastaPB1, "PA": fastaPA, "HA": fastaHA, "NP": fastaNP, "NA": fastaNA, "MP": fastaMP, "NS": fastaNS}

    # Merge on 
    merged_dict = {}
    
    for key, f in dictionary.items():
        # f["Isolate_Name"] = f["header"].astype(str).str.replace(">", "")
        # merged = f.merge(metadata, on = "Isolate_Name", how = "inner")
        f["Isolate_Id"] = f["header"].astype(str).str.replace(">", "")
        merged = f.merge(metadata, on = "Isolate_Id", how = "inner")
        
        merged["new-header"] = ">"+ merged["newid"]
        merged_dict[key] = merged        

    # Print new FASTA with newid as the header
    for segment, df in merged_dict.items():
        try:  
            os.mkdir("./alignments/2026-03-16/")
    
        except OSError as error:
            pass
    
        with open(f"./alignments/2026-03-16/new-headers_{segment}_from_tufts.fasta", "w") as f:
            for index, row in df.iterrows():
                f.write(f"{row['new-header']}\n")
                f.write(f"{row['sequence']}\n")

In [494]:
FILE = "./metadata/updated_metadata_cleaned.csv"

correct_headers(FILE)